# Learning Theory — Exercises

Companion to the note [Learning Theory](Learning%20Theory.md).

Practise empirical vs. true risk, PAC sample complexity for finite classes, Hoeffding and the union bound, VC dimension and shattering, Rademacher complexity, margins and the modern double-descent puzzle: first by hand, then with small NumPy simulations that make the bounds visible.

**15 exercises** · 🧠 Concept ×4 · ✍️ By hand ×6 · 💻 Code ×5

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from itertools import product
from scipy.optimize import linprog
from scipy.stats import norm
rng = np.random.default_rng(7)

## Part A · Concepts

### Exercise 1 · Empirical vs. true risk
*🧠 Concept · ★☆☆*

The note defines $\hat R(h)=\frac1N\sum_i\ell(h(x_i),y_i)$ and $R(h)=\mathbb E[\ell(h(x),y)]$.

1. For a **fixed** $h$ chosen before seeing the data, is $\hat R(h)$ an unbiased estimate of $R(h)$?
2. For the ERM solution $\hat h=\arg\min_{h\in\mathcal H}\hat R(h)$, why is $\hat R(\hat h)$ typically *smaller* than $R(\hat h)$?
3. What, then, does a generalisation bound have to control?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Linearity of expectation answers 1. For 2, think of picking the best of many noisy coins: the selection itself uses the noise.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Yes: $\mathbb E[\hat R(h)] = \frac1N\sum_i \mathbb E[\ell(h(x_i),y_i)] = R(h)$ because the samples are i.i.d. and $h$ does not depend on them.
2. $\hat h$ is chosen *because* its empirical risk is low, so it is selected partly for fitting the noise in this particular sample.
   $\mathbb E[\hat R(\hat h)] \le \mathbb E[\min_h \ldots]$ is biased downwards (optimistic), the same reason the test set must not be used for tuning ([[Common Pitfalls]]).
3. A bound must hold **uniformly** over the class: $\sup_{h\in\mathcal H}|R(h)-\hat R(h)|$. That is why capacity measures ($\ln|\mathcal H|$, VC dimension, Rademacher complexity) appear in every bound.

</details>

### Exercise 2 · No Free Lunch and inductive bias
*🧠 Concept · ★☆☆*

The No-Free-Lunch theorem says that averaged over *all* possible labelling functions, every learner has the same expected
off-training-set error. Explain in two or three sentences why this does **not** make machine learning pointless, and name the
inductive bias of (a) $k$-NN, (b) linear regression, (c) a CNN.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Real-world problems are not drawn uniformly from all functions. Which kind of functions does each model prefer?

</details>

<details>
<summary><b>✅ Solution</b></summary>

NFL averages over a uniform distribution on all labelings, most of which are pure noise with no structure. Real problems are
highly structured, so a learner whose **inductive bias** matches that structure wins on them (and loses on problems we do not care about).
(a) $k$-NN: nearby points have similar labels (smoothness / locality). (b) Linear regression: the target is (close to) linear in the features.
(c) CNN: locality and translation equivariance via weight sharing ([[CNN]]).

</details>

### Exercise 3 · Approximation vs. estimation error
*🧠 Concept · ★★☆*

Let $h^\star$ be the Bayes-optimal predictor, $h_{\mathcal H}=\arg\min_{h\in\mathcal H}R(h)$ and $\hat h$ the ERM output. Write
$R(\hat h)-R(h^\star)$ as a sum of two terms, name them, and say how each one changes when (a) $\mathcal H$ grows and (b) $N$ grows.
Relate this to [[Bias-Variance Tradeoff]].

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Add and subtract $R(h_{\mathcal H})$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$R(\hat h)-R(h^\star) = \underbrace{R(h_{\mathcal H})-R(h^\star)}_{\text{approximation error}} + \underbrace{R(\hat h)-R(h_{\mathcal H})}_{\text{estimation error}}$.

(a) Larger $\mathcal H$: approximation error falls (or stays), estimation error rises (more ways to overfit).
(b) Larger $N$: approximation error is unchanged (it does not depend on data), estimation error falls (roughly like $\sqrt{\mathrm{capacity}/N}$).
This is the **bias–complexity trade-off**: approximation error plays the role of bias, estimation error that of variance.

</details>

### Exercise 4 · Modern puzzles
*🧠 Concept · ★★☆*

A ResNet with $25$M parameters is trained on $50{,}000$ images to zero training error, and still has low test error.
Its VC dimension is far larger than $N$, so the VC bound is vacuous. Give **three** explanations from the note's "modern puzzles"
list for why it generalises anyway, and describe what a *double descent* curve looks like.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Which solution among the many interpolating ones does SGD find? Which capacity measure depends on the data or the weights rather than on the parameter count?

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **Implicit regularisation of SGD**: among all interpolating solutions, (S)GD from small initialisation tends to find low-norm / flat ones,
   so the *effective* capacity is much smaller than the parameter count.
2. **Data- and norm-dependent bounds** (Rademacher complexity, margin bounds) can be small even when the VC dimension is huge.
3. **Neural tangent kernel**: very wide networks behave like kernel regression with a fixed kernel, whose minimum-norm interpolant generalises
   under suitable conditions.

**Double descent**: test error first follows the classical U-curve as capacity grows, peaks at the *interpolation threshold* (parameters ≈ samples),
then decreases again in the over-parameterised regime. You will reproduce it in Exercise 15.

</details>

## Part B · By hand

### Exercise 5 · Sample complexity of a finite class
*✍️ By hand · ★☆☆*

Realisable case, finite class: $m\ge\frac{1}{\epsilon}\big(\ln|\mathcal H|+\ln\frac1\delta\big)$.
With $|\mathcal H|=1000$, $\epsilon=0.05$ and $\delta=0.01$, what is the smallest integer $m$ that guarantees the PAC criterion?

In [ ]:
m_finite = None  # an integer

check('m (finite class)', m_finite, np.ceil((np.log(1000) + np.log(1/0.01)) / 0.05))

<details>
<summary><b>💡 Hint 1</b></summary>

$\ln 1000\approx 6.908$ and $\ln 100\approx 4.605$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Divide the sum by $0.05$ (i.e. multiply by 20) and round **up**.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$m \ge 20\,(6.908 + 4.605) = 20\cdot 11.513 = 230.26$, so $m = 231$.
Note the dependence is only **logarithmic** in $|\mathcal H|$ and $1/\delta$ but linear in $1/\epsilon$.

```python
m_finite = 231
```

</details>

### Exercise 6 · Deriving the finite-class bound
*✍️ By hand · ★★☆*

Prove the bound of the previous exercise. Assume some $h\in\mathcal H$ has zero true risk (realisable) and ERM returns *any*
hypothesis consistent with the $m$ training samples. Call $h$ **bad** if $R(h)>\epsilon$.

1. Bound the probability that a single fixed bad $h$ is consistent with all $m$ samples.
2. Use the union bound over $\mathcal H$ and $1-\epsilon\le e^{-\epsilon}$ to finish.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

A bad $h$ is correct on one random sample with probability $<1-\epsilon$; samples are independent.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $P(h \text{ consistent}) = \prod_{i=1}^m P(h(x_i)=y_i) < (1-\epsilon)^m \le e^{-\epsilon m}$.
2. $P(\exists\text{ bad consistent } h) \le \sum_{h\text{ bad}} P(h\text{ consistent}) \le |\mathcal H|e^{-\epsilon m}$.
   Requiring $|\mathcal H|e^{-\epsilon m}\le\delta$ and solving for $m$ gives $m\ge\frac1\epsilon\left(\ln|\mathcal H|+\ln\frac1\delta\right)$.
   With probability $\ge 1-\delta$, every consistent hypothesis (in particular the ERM output) has $R(h)\le\epsilon$. ∎

</details>

### Exercise 7 · Hoeffding: how big a test set?
*✍️ By hand · ★☆☆*

For a fixed classifier and 0–1 loss, Hoeffding's inequality gives
$P\big(|\hat R(h)-R(h)|>\epsilon\big)\le 2e^{-2N\epsilon^2}$.
How many **test** samples do you need so that the test error is within $\epsilon=0.05$ of the true error with probability at least $95\%$?

In [ ]:
n_test = None  # smallest integer

check('test-set size', n_test, np.ceil(np.log(2/0.05) / (2*0.05**2)))

<details>
<summary><b>💡 Hint</b></summary>

Set $2e^{-2N\epsilon^2}=\delta$ and solve: $N = \frac{\ln(2/\delta)}{2\epsilon^2}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$N \ge \frac{\ln(2/0.05)}{2\cdot 0.05^2} = \frac{\ln 40}{0.005} = \frac{3.689}{0.005} = 737.8$, so $N=738$.
This only holds because $h$ is fixed **before** looking at the test set; if you pick among many models on the test set you need a union bound (next exercise).

```python
n_test = 738
```

</details>

### Exercise 8 · Agnostic bound with the union bound
*✍️ By hand · ★★☆*

Now we do not assume realisability. Combine Hoeffding with a union bound over a finite class to get the **uniform** statement
$P\big(\exists h\in\mathcal H: |\hat R(h)-R(h)|>\epsilon\big)\le 2|\mathcal H|e^{-2N\epsilon^2}$.
Solve for $N$ and evaluate it for $|\mathcal H| = 1024$, $\epsilon=0.1$, $\delta=0.05$.

In [ ]:
n_agnostic = None  # smallest integer

check('agnostic sample size', n_agnostic, np.ceil((np.log(1024) + np.log(2/0.05)) / (2*0.1**2)))

<details>
<summary><b>💡 Hint</b></summary>

$N\ge\frac{\ln|\mathcal H|+\ln(2/\delta)}{2\epsilon^2}$, with $\ln 1024 = 10\ln 2\approx 6.931$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$N\ge\frac{6.931+3.689}{0.02}=\frac{10.620}{0.02}=531.0$, so $N=532$.
Compared with the realisable case the dependence on $\epsilon$ worsens from $1/\epsilon$ to $1/\epsilon^2$.

```python
n_agnostic = 532
```

</details>

### Exercise 9 · VC dimension of simple classes
*✍️ By hand · ★★☆*

Find the VC dimension of each class on $\mathbb R$ (store the numbers):

(a) thresholds $h_t(x)=\mathbb 1[x\ge t]$,
(b) intervals $h_{a,b}(x)=\mathbb 1[a\le x\le b]$,
(c) linear classifiers $\mathrm{sign}(w^\top x+b)$ in $\mathbb R^2$ (the note's formula $d+1$).

For each, give a set of that size that is shattered and argue why no larger set is.

In [ ]:
vc_threshold = None
vc_interval = None
vc_linear_2d = None

check('VC thresholds', vc_threshold, 1)
check('VC intervals', vc_interval, 2)
check('VC linear in R^2', vc_linear_2d, 3)

<details>
<summary><b>💡 Hint 1</b></summary>

For two points $x_1<x_2$, can a threshold produce the labels $(1,0)$?

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

For three points $x_1<x_2<x_3$, can an interval produce $(1,0,1)$?

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **1**: any single point can be labelled 0 or 1, but for $x_1<x_2$ the labelling $(1,0)$ is impossible because thresholds are monotone.
(b) **2**: two points can get all four labellings, but for $x_1<x_2<x_3$ the labelling $(1,0,1)$ needs a gap inside the interval.
(c) **3**: three non-collinear points are shattered by lines; for any 4 points either one lies in the convex hull of the other three
or they form a convex quadrilateral, and the "XOR" diagonal labelling cannot be separated (Radon's theorem). In general VC $=d+1$.

```python
vc_threshold = 1
vc_interval = 2
vc_linear_2d = 3
```

</details>

### Exercise 10 · Margins: a dimension-free bound
*✍️ By hand · ★★☆*

The perceptron mistake bound says: if all $\lVert x_i\rVert\le R$ and some unit vector $w^\star$ separates the data with margin
$y_i\,w^{\star\top}x_i\ge\gamma$, the perceptron makes at most $(R/\gamma)^2$ mistakes.

1. With $R=10$ and $\gamma=2$, what is the bound?
2. Why does this bound (and the SVM margin bound in the note) not depend on the input dimension $d$, and why does that explain SVMs with kernels in infinite-dimensional feature spaces?

In [ ]:
mistake_bound = None

check('perceptron mistake bound', mistake_bound, (10/2)**2)

<details>
<summary><b>💡 Hint</b></summary>

Track $w_t^\top w^\star$ (grows by at least $\gamma$ per mistake) and $\lVert w_t\rVert^2$ (grows by at most $R^2$ per mistake).

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $(10/2)^2 = 25$.
2. The proof only uses inner products, $R$ and $\gamma$: after $k$ mistakes $k\gamma\le w_k^\top w^\star\le\lVert w_k\rVert\le\sqrt k R$, so $k\le R^2/\gamma^2$.
   No coordinate count appears. Similarly, the fat-shattering dimension of large-margin linear classifiers is $\lesssim R^2/\gamma^2$,
   so an SVM in a huge (even infinite) kernel feature space can generalise as long as the margin is large relative to the data radius
   ([[Support Vector Machines]]).

```python
mistake_bound = 25
```

</details>

## Part C · Simulations in NumPy

### Exercise 11 · Hoeffding in action
*💻 Code · ★☆☆*

A classifier has true error $R=0.3$. Simulate `trials=20000` test sets of size $N=200$ (each test error is a mean of Bernoulli$(0.3)$ draws).
Estimate $P(|\hat R - R| > 0.05)$ and store it in `p_emp`; store the Hoeffding bound $2e^{-2N\epsilon^2}$ in `p_bound`.
Is the bound tight?

In [ ]:
R_true, N, eps, trials = 0.3, 200, 0.05, 20000
p_emp = None
p_bound = None

if p_emp is not None:
    _exact = 1 - (norm.cdf(eps, 0, np.sqrt(R_true*(1-R_true)/N)) - norm.cdf(-eps, 0, np.sqrt(R_true*(1-R_true)/N)))
    check('empirical prob. close to the CLT value', p_emp, _exact, tol=0.03)
    check('Hoeffding bound value', p_bound, 2*np.exp(-2*N*eps**2))
    check('bound holds', p_emp <= p_bound, True)
else:
    check('empirical probability', None, 0)

<details>
<summary><b>💡 Hint</b></summary>

`rng.binomial(N, R_true, size=trials) / N` gives all test errors at once.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The empirical probability is about $0.10$–$0.11$ while the bound is $2e^{-1}\approx 0.74$: valid but loose, because Hoeffding only uses the range $[0,1]$
and not the actual variance $R(1-R)$. Bernstein-type inequalities use the variance and are tighter.

```python
R_true, N, eps, trials = 0.3, 200, 0.05, 20000
errs = rng.binomial(N, R_true, size=trials) / N
p_emp = np.mean(np.abs(errs - R_true) > eps)
p_bound = 2 * np.exp(-2 * N * eps**2)
print(p_emp, p_bound)
```

</details>

### Exercise 12 · Brute-force shattering
*💻 Code · ★★☆*

Write `is_shattered(P)` that returns `True` if linear classifiers $\mathrm{sign}(w^\top x+b)$ can realise **every** labelling
$y\in\{-1,+1\}^n$ of the points in `P` (shape $(n,d)$). For each labelling, test linear separability with an LP feasibility problem:
find $(w,b)$ with $y_i(w^\top x_i+b)\ge 1$ (use `scipy.optimize.linprog` with zero objective).
Verify VC $=3$ in $\mathbb R^2$ on the given point sets.

In [ ]:
def is_separable(P, y):
    # TODO: LP feasibility; return True/False
    return None

def is_shattered(P):
    # TODO: loop over all 2^n labelings
    return None

tri = np.array([[0., 0.], [1., 0.], [0., 1.]])          # 3 points, not collinear
line3 = np.array([[0., 0.], [1., 0.], [2., 0.]])        # 3 collinear points
square = np.array([[0., 0.], [1., 0.], [0., 1.], [1., 1.]])

check('triangle shattered', is_shattered(tri), True)
check('collinear triple not shattered', is_shattered(line3), False)
check('square not shattered', is_shattered(square), False)

<details>
<summary><b>💡 Hint 1</b></summary>

Constraint $y_i(w^\top x_i + b)\ge 1$ becomes `A_ub @ z <= b_ub` with rows $-y_i\,[x_i, 1]$ and right-hand side $-1$; $z=(w,b)$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Pass `bounds=[(None, None)] * (d+1)` (variables are free) and check `res.status == 0` (feasible).

</details>

<details>
<summary><b>✅ Solution</b></summary>

A triangle admits all 8 labellings. For collinear points the labelling $(+,-,+)$ fails, and for the square the XOR labelling fails.
So *some* set of size 3 is shattered (VC $\ge3$) but no set of size 4 is (proved in Exercise 9), hence VC $=3=d+1$.
Note the definition: VC needs **one** shattered set of size $d$, not all of them (the collinear triple is irrelevant).

```python
def is_separable(P, y):
    n, d = P.shape
    A = -y[:, None] * np.c_[P, np.ones(n)]
    res = linprog(np.zeros(d + 1), A_ub=A, b_ub=-np.ones(n), bounds=[(None, None)] * (d + 1), method='highs')
    return res.status == 0

def is_shattered(P):
    return all(is_separable(P, np.array(lab, dtype=float)) for lab in product([-1, 1], repeat=len(P)))

tri = np.array([[0., 0.], [1., 0.], [0., 1.]])
line3 = np.array([[0., 0.], [1., 0.], [2., 0.]])
square = np.array([[0., 0.], [1., 0.], [0., 1.], [1., 1.]])
```

</details>

### Exercise 13 · Rademacher complexity of thresholds
*💻 Code · ★★★*

The empirical Rademacher complexity of a class on a sample $x_1,\dots,x_n$ is
$\hat{\mathfrak R}_n(\mathcal H)=\mathbb E_\sigma\Big[\sup_{h\in\mathcal H}\frac1n\sum_i\sigma_i h(x_i)\Big]$,
$\sigma_i\in\{\pm1\}$ uniform. For thresholds $h_t(x)=+1$ if $x\ge t$ else $-1$, the labellings on sorted points are
"first $k$ are $-1$, the rest $+1$" for $k=0,\dots,n$.

Implement `rademacher_thresholds(x, n_draws, seed)` by Monte Carlo. Then compute it for $n\in\{10,100,1000\}$ (store in `rad`, a dict)
and compare with Massart's bound $\sqrt{2\ln(n+1)/n}$ (the class induces $n+1$ labellings).

In [ ]:
def rademacher_thresholds(x, n_draws=2000, seed=0):
    # TODO: Monte Carlo estimate of E_sigma sup_h (1/n) sum sigma_i h(x_i)
    return None

rad = None   # {10: ..., 100: ..., 1000: ...}

def _ref_rad(x, n_draws=400, seed=1):
    r = np.random.default_rng(seed); n = len(x)
    H = np.where(np.arange(n)[None, :] >= np.arange(n + 1)[:, None], 1.0, -1.0)   # all n+1 labelings (sorted x)
    return np.mean([np.max(H @ r.choice([-1, 1], n)) / n for _ in range(n_draws)])
_xs = np.sort(np.random.default_rng(3).uniform(size=50))
check('matches brute force (n=50)', rademacher_thresholds(_xs, 4000, 0), _ref_rad(_xs), tol=0.02)
check('n=1 gives exactly 1', rademacher_thresholds(np.array([0.5]), 200, 0), 1.0)
if rad is not None:
    check('below Massart bound for all n', all(rad[n] <= np.sqrt(2*np.log(n+1)/n) for n in rad), True)
    check('decreases with n', rad[10] > rad[100] > rad[1000], True)
else:
    check('below Massart bound for all n', None, True)

<details>
<summary><b>💡 Hint 1</b></summary>

The value of labeling $k$ is $\frac1n\big(-\sum_{i<k}\sigma_i+\sum_{i\ge k}\sigma_i\big)=\frac1n\big(S-2P_k\big)$ with $S=\sum_i\sigma_i$ and prefix sum $P_k$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

So $\sup_k = \frac1n(S - 2\min_k P_k)$ with $P_0=0$: one `np.cumsum` per draw (or vectorised over draws).

</details>

<details>
<summary><b>✅ Solution</b></summary>

For sorted points only the order matters, so the estimate does not depend on where the $x_i$ are. With the prefix-sum trick the supremum costs $O(n)$.
Typical values: $\hat{\mathfrak R}\approx 0.42, 0.15, 0.05$ for $n=10,100,1000$, roughly $\propto 1/\sqrt n$ and below Massart's
$\sqrt{2\ln(n+1)/n}\approx 0.69, 0.30, 0.12$. The generalisation bound $R(h)\le\hat R(h)+2\mathfrak R_n+O(\sqrt{\ln(1/\delta)/n})$ is therefore informative here.

```python
def rademacher_thresholds(x, n_draws=2000, seed=0):
    r = np.random.default_rng(seed)
    n = len(x)
    sig = r.choice([-1, 1], size=(n_draws, n))
    P = np.c_[np.zeros(n_draws), np.cumsum(sig, axis=1)]   # P_0 .. P_n
    S = P[:, -1]
    return np.mean((S - 2 * P.min(axis=1)) / n)

rad = {n: rademacher_thresholds(np.sort(rng.uniform(size=n))) for n in (10, 100, 1000)}
print({n: round(v, 3) for n, v in rad.items()})
```

</details>

### Exercise 14 · ERM and the generalisation gap
*💻 Code · ★★☆*

Data: $x\sim U[0,1]$, $y=\mathbb 1[x\ge 0.6]$ with 15% label noise. Hypothesis class: thresholds on the grid $t\in\{0,0.01,\dots,1\}$ ($|\mathcal H|=101$).
For $N\in\{10,30,100,300,1000\}$, repeat 200 times: draw a training set, run ERM, and record the gap $R(\hat h)-\hat R(\hat h)$
(the true risk of threshold $t$ is $0.15+0.7\,|t-0.6|$). Store the mean gaps in `gaps` (a list in the order of $N$).
Compare with the agnostic union bound $\sqrt{\ln(2|\mathcal H|/\delta)/(2N)}$ at $\delta=0.05$.

In [ ]:
Ns = [10, 30, 100, 300, 1000]
grid = np.linspace(0, 1, 101)
def true_risk(t):
    return 0.15 + 0.7 * np.abs(t - 0.6)
gaps = None

if gaps is not None:
    check('gap shrinks with N', gaps[0] > gaps[2] > gaps[-1], True)
    check('gap is clearly positive for small N (ERM is optimistic)', gaps[0] > 0.05, True)
    check('gap below the uniform bound', all(g <= np.sqrt(np.log(2*101/0.05)/(2*n)) for g, n in zip(gaps, Ns)), True)
else:
    check('gap shrinks with N', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Vectorise: `pred = x[:, None] >= grid[None, :]` gives an $(N,101)$ matrix; the empirical risks are `(pred != y[:, None]).mean(0)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The mean gap is clearly positive for small $N$ (about $0.14$ at $N=10$: ERM is optimistic, Exercise 1) and shrinks quickly towards 0 (below $0.01$ for $N\ge 100$). The uniform bound is valid but loose by a factor of a few,
which is typical: union bounds treat all 101 thresholds as independent, while neighbouring thresholds make nearly the same predictions.

```python
Ns = [10, 30, 100, 300, 1000]
grid = np.linspace(0, 1, 101)
def true_risk(t):
    return 0.15 + 0.7 * np.abs(t - 0.6)
gaps = []
for N in Ns:
    g = []
    for _ in range(200):
        x = rng.uniform(size=N)
        y = (x >= 0.6) ^ (rng.uniform(size=N) < 0.15)
        emp = ((x[:, None] >= grid[None, :]) != y[:, None]).mean(0)
        j = np.argmin(emp)
        g.append(true_risk(grid[j]) - emp[j])
    gaps.append(np.mean(g))
print(np.round(gaps, 3))
```

</details>

### Exercise 15 · Double descent with random features
*💻 Code · ★★★*

Reproduce double descent. Use $n=40$ training points, $x\in\mathbb R^5$, $y = \sin(x_1)+x_2^2/2+\varepsilon$ ($\sigma=0.3$), and a test set of 1000.
Features: $\phi(x)=\mathrm{ReLU}(Vx)$ with $V\in\mathbb R^{p\times5}$ Gaussian. Fit the **minimum-norm least-squares** solution `w = np.linalg.pinv(Phi) @ y`.
For $p\in\{5,10,20,30,38,40,42,50,80,160,640\}$ average the test MSE over 20 seeds and store a dict `test_mse`.
Where is the peak? What happens after it?

In [ ]:
p_list = [5, 10, 20, 30, 38, 40, 42, 50, 80, 160, 640]
def make_data(n, r):
    X = r.normal(size=(n, 5))
    return X, np.sin(X[:, 0]) + X[:, 1]**2 / 2 + 0.3 * r.normal(size=n)
test_mse = None   # {p: mean test MSE}

if test_mse is not None:
    check('peak near p = n', test_mse[40] > 3 * max(test_mse[10], test_mse[640]), True)
    check('second descent', test_mse[640] < test_mse[50], True)
else:
    check('peak near p = n', None, True)

<details>
<summary><b>💡 Hint 1</b></summary>

At $p=n$ the square matrix $\Phi$ is just barely invertible: its smallest singular value is tiny, so $\lVert w\rVert$ explodes.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Use one `np.random.default_rng(seed)` per repetition for data **and** $V$; compute $\Phi$ for train and test with the same $V$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Test error first improves, then explodes at the **interpolation threshold** $p\approx n=40$, where the minimum-norm interpolant must use enormous
weights to fit the noise exactly. Beyond it, many interpolating solutions exist and `pinv` picks the one with the smallest norm; as $p$ grows that norm
(and hence the variance) shrinks, so the error descends a second time. Adding a small ridge penalty removes the peak, which shows that the
spike is a variance effect of unregularised interpolation, not a law of over-parameterisation.

```python
p_list = [5, 10, 20, 30, 38, 40, 42, 50, 80, 160, 640]
def make_data(n, r):
    X = r.normal(size=(n, 5))
    return X, np.sin(X[:, 0]) + X[:, 1]**2 / 2 + 0.3 * r.normal(size=n)
test_mse = {}
for p in p_list:
    errs = []
    for seed in range(20):
        r = np.random.default_rng(seed)
        Xtr, ytr = make_data(40, r)
        Xte, yte = make_data(1000, r)
        V = r.normal(size=(p, 5)) / np.sqrt(5)
        Ptr, Pte = np.maximum(Xtr @ V.T, 0), np.maximum(Xte @ V.T, 0)
        w = np.linalg.pinv(Ptr) @ ytr
        errs.append(np.mean((Pte @ w - yte) ** 2))
    test_mse[p] = np.mean(errs)
print({p: round(v, 2) for p, v in test_mse.items()})
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Learning Theory](Learning%20Theory.md).*